# IS630: Green Light for an Indie Studio
## Notebook 00: Setup, Acquisition, Join, Scope and Train/Test Split

> **AI Coding Assistant Declaration**:
> In accordance with course guidelines, an AI coding assistant was used strictly for code scaffolding and data preparation. All code blocks below are labeled as AI-assisted. All modelling decisions, interpretations, and written commentaries are independently authored by group members in the designated sections below.

| Dataset | Kaggle ref | Version | Pulled |
|---|---|---|---|
| Steam | `fronkongames/steam-games-dataset` | v49 | 19 Sep 2026 |
| HowLongToBeat | `b4n4n4p0wer/how-long-to-beat-video-game-playtime-dataset` | v5 | 19 Sep 2026 |

The Steam dataset refreshes continuously. Do not re-download mid-project.

**Reads:** `data/raw/steam_v49.zip`, `data/raw/hltb_v5.zip`
**Writes:** `00_steam.parquet`, `00_hltb.parquet`, `00_joined.parquet`, `00_train.parquet`, `00_test_holdout.parquet`, all in `data/interim`

## 1. Environment setup and data acquisition
*Code: Generated by AI Assistant*

In [1]:
import sys
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists()) / "src"))
from is630 import config
from is630.clean import loose_title, normalise_title, prefix_columns
from is630.features import classify_story_driven

TEST_FRACTION = 0.20

In [2]:
# Download only if the pinned archive is missing, then verify the SHA-256 checksum of the CSV inside it
archives = [
    (config.STEAM_ZIP, config.STEAM_REF, config.STEAM_VERSION, config.STEAM_CSV, config.STEAM_SHA256),
    (config.HLTB_ZIP, config.HLTB_REF, config.HLTB_VERSION, config.HLTB_CSV, config.HLTB_SHA256),
]
for path, ref, ver, member, sha256 in archives:
    if not path.exists():
        url = f"https://www.kaggle.com/api/v1/datasets/download/{ref}?datasetVersionNumber={ver}"
        urllib.request.urlretrieve(url, path)
    digest = config.verify_sha256(path, member, sha256)
    print(f"{path.name}/{member} sha256={digest[:12]}... verified")

steam_v49.zip/games.csv sha256=1b48008b01a7... verified
hltb_v5.zip/hltb_dataset_normalized.csv sha256=59b91b918e68... verified


## 2. Steam dataset
*Code: Generated by AI Assistant*

*Written by AI Assistant*: the CSV header fuses `Discount` and `DLC count` into one name, giving 39 headers for 40 fields. Reading it as-is shifts every column one place, so the header is split before loading.

In [3]:
with zipfile.ZipFile(config.STEAM_ZIP) as z:
    with z.open(config.STEAM_CSV) as f:
        header = pd.read_csv(f, nrows=0).columns.tolist()

i = header.index("DiscountDLC count")
names = header[:i] + ["Discount", "DLC count"] + header[i + 1:]

with zipfile.ZipFile(config.STEAM_ZIP) as z:
    with z.open(config.STEAM_CSV) as f:
        steam = pd.read_csv(f, header=0, names=names, index_col=False)

# Stamp every column with its source. Both datasets carry a name, a release
# date, a genre list and hour-valued columns, so the prefix is what keeps them
# apart once they are joined.
steam = prefix_columns(steam, "steam")

print(f"steam: {steam.shape[0]:,} rows, {steam.shape[1]} columns")
steam.head()

steam: 125,855 rows, 40 columns


,steam_appid,steam_name,steam_release_date,steam_estimated_owners,steam_peak_ccu,steam_required_age,steam_price,steam_discount,steam_dlc_count,steam_about_the_game,...,steam_average_playtime_two_weeks,steam_median_playtime_forever,steam_median_playtime_two_weeks,steam_developers,steam_publishers,steam_categories,steam_genres,steam_tags,steam_screenshots,steam_movies
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,...,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",...,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...,NaN
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...",...,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...,NaN
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,0,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",...,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
4,3631080,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,0,4.99,0,0,Its not just a Maze; its a Quest! Enter the ca...,...,0,0,0,Reality Expanded LLC,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN


In [4]:
print("Missing values, columns with any:")
print(steam.isna().sum()[lambda v: v > 0].sort_values(ascending=False))
print("\nZero-filled rather than null:")
for col in ["steam_metacritic_score", "steam_price", "steam_positive", "steam_negative", "steam_median_playtime_forever"]:
    print(f"  {col}: {(steam[col] == 0).sum():,} zeros")

Missing values, columns with any:
steam_movies            125855
steam_score_rank        125815
steam_metacritic_url    121597
steam_reviews           113674
steam_notes             102545
steam_website            75335
steam_support_url        70560
steam_tags               42502
steam_support_email      22630
steam_categories          8966
steam_publishers          8922
steam_about_the_game      8462
steam_developers          8449
steam_genres              8423
steam_screenshots         6027
steam_header_image          81
steam_name                   1
dtype: int64

Zero-filled rather than null:
  steam_metacritic_score: 121,597 zeros
  steam_price: 26,661 zeros
  steam_positive: 45,509 zeros
  steam_negative: 58,432 zeros
  steam_median_playtime_forever: 99,682 zeros


### Interpretation: Steam data inspection
**Author**: Lee Sin Yang 
<!-- Record your observations on the Steam table: row and column counts, which columns are missing values and whether that missingness looks random, which columns are zero-filled rather than null and what that hides, and which columns the three analytical questions will actually need. -->
Steam df has 40 columns and 125855 rows. Missingness varies across columns, with some columns such as the publishers, developers and about the game in the thousands, while movies, metacritic url and reviews has high missingness. This is likely due to most games not actually being very popular. 

## 3. HowLongToBeat dataset
*Code: Generated by AI Assistant*

In [5]:
with zipfile.ZipFile(config.HLTB_ZIP) as z:
    with z.open(config.HLTB_CSV) as f:
        hltb = pd.read_csv(f)

hltb = prefix_columns(hltb, "hltb")

print(f"hltb: {hltb.shape[0]:,} rows, {hltb.shape[1]} columns")
hltb.head()

hltb: 166,754 rows, 28 columns


,hltb_id,hltb_name,hltb_type,hltb_platform,hltb_genres,hltb_developer,hltb_publisher,hltb_release_date,hltb_release_precision,hltb_release_year,...,hltb_all_styles_polled,hltb_all_styles,hltb_single_player_polled,hltb_single_player,hltb_co_op_polled,hltb_co_op,hltb_versus_polled,hltb_versus,hltb_source_url,hltb_crawled_at
0,1,688(I) Hunter/Killer,game,PC,"Simulation, Strategy/Tactical",Sonalysts,Electronic Arts,1997-07-04,day,1997.0,...,15.0,15.93,8.0,10.88,NaN,NaN,NaN,NaN,https://howlongtobeat.com/game/1,2026-03-27T10:06:13Z
1,2,Beyond Good & Evil 2,game,NaN,"Action, Adventure",Ubisoft Montpellier,Ubisoft,0000-00-00,day,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://howlongtobeat.com/game/2,2026-03-27T10:06:14Z
2,18,'Splosion Man,game,Xbox 360,Action,Twisted Pixel Games,Microsoft Games Studios,2009-07-22,day,2009.0,...,62.0,8.76,32.0,7.80,1.0,9.0,NaN,NaN,https://howlongtobeat.com/game/18,2026-03-27T10:06:16Z
3,19,.hack//G.U. Vol. 1: Rebirth,game,"Nintendo Switch, PC, PlayStation 2, PlayStation 4","Action, Role-Playing",CyberConnect2,Bandai,2006-05-18,day,2006.0,...,181.0,23.41,76.0,18.52,NaN,NaN,NaN,NaN,https://howlongtobeat.com/game/19,2026-03-27T10:06:15Z
4,20,.hack//G.U. Vol. 2: Reminisce,game,"Nintendo Switch, PC, PlayStation 2, PlayStation 4","Action, Role-Playing",CyberConnect2,Bandai,2006-09-28,day,2006.0,...,88.0,26.62,41.0,19.82,NaN,NaN,NaN,NaN,https://howlongtobeat.com/game/20,2026-03-27T10:06:15Z


In [6]:
print("Entry types:")
print(hltb["hltb_type"].value_counts())
print(f"\nmain_story missing: {hltb['hltb_main_story'].isna().sum():,} of {len(hltb):,}")
print(f"release_year missing: {hltb['hltb_release_year'].isna().sum():,} of {len(hltb):,}")

Entry types:
hltb_type
game             155586
dlc/expansion      2728
omit               2587
multi              2249
delay               894
compil              691
sports              645
mod                 516
endless             454
hack                404
Name: count, dtype: int64

main_story missing: 120,300 of 166,754
release_year missing: 70,593 of 166,754


### Interpretation: HowLongToBeat data inspection
**Author**: Lee Sin Yang 

The HLTB dataset has 28 columns and 166754 rows. Missingness varies across columns, as many fields are optional and only applicable to certain types of games.

## 4. Save the raw tables
*Code: Generated by AI Assistant*

*Written by AI Assistant*: both tables are saved as loaded. Column pruning and type cleaning belong to later stages.

In [7]:
steam.to_parquet(config.INTERIM / "00_steam.parquet", index=False)
hltb.to_parquet(config.INTERIM / "00_hltb.parquet", index=False)
print(f"saved unmodified: {len(steam):,} steam rows, {len(hltb):,} hltb rows")

saved unmodified: 125,855 steam rows, 166,754 hltb rows


## 5. Joining the two datasets
*Code: Generated by AI Assistant*

*Written by AI Assistant*: no shared identifier exists, so the key is the normalised title. Release year is recorded rather than used as part of the key, because HowLongToBeat stores a game's original release year while Steam stores its Steam release date. Requiring the two to agree would drop console-first ports and re-releases, which are among the best covered titles.

Both sides are reduced to one row per title first, so the join cannot multiply rows:

- HowLongToBeat is filtered to `type == "game"` with a positive `main_story`. A release year of 0 is the source's placeholder for an unknown date (`0000-00-00`) and is set to missing before matching.
- Steam keeps, per title, the entry with the most reviews.

The match runs in two passes. Each Steam game ends up with at most one HowLongToBeat entry and each entry with at most one Steam game.

1. Exact pass on `title_key`. Where a title has several candidates, the one whose release year is closest to the Steam release year is kept, breaking ties on the larger `main_story_polled`.
2. Fallback pass on `loose_title` (see `src/is630/clean.py`), for Steam titles the exact pass left unmatched and HowLongToBeat entries it left unused. The fallback key drops accents, reads `&` as "and", and strips a trailing `(2005)` and edition suffixes such as "GOTY Edition", "Definitive Edition" or "Director's Cut". The same closest-year rule picks the candidate. If two Steam games reach the same entry, the one with the smaller year gap keeps it, then the one with more reviews.

The join is a left join, so every Steam game survives. `matched_hltb` marks the ones that found a partner and `match_pass` records which pass found it (1 exact, 2 fallback, missing if unmatched), so the fallback matches can be dropped in a sensitivity check. `year_diff` is the Steam year minus the HowLongToBeat year, for tightening the match later.

In [8]:
steam["title_key"] = normalise_title(steam["steam_name"])
steam["steam_year"] = pd.to_datetime(steam["steam_release_date"], errors="coerce", format="mixed").dt.year

hltb["title_key"] = normalise_title(hltb["hltb_name"])
# 0 is the source's placeholder for an unknown date, not a year
print(f"hltb release_year == 0 set to missing: {int(hltb['hltb_release_year'].eq(0).sum()):,}")
hltb["hltb_release_year"] = hltb["hltb_release_year"].where(hltb["hltb_release_year"].gt(0))

candidates = hltb[hltb["hltb_type"].eq("game") & hltb["hltb_main_story"].notna() & hltb["hltb_main_story"].gt(0)].copy()
print(f"hltb candidates with a main story time: {len(candidates):,}")

# Deterministic tie-break on steam_appid so deduplication is identical across pandas versions
steam_one = (
    steam.assign(_reviews=steam["steam_positive"] + steam["steam_negative"])
    .sort_values(["_reviews", "steam_appid"], ascending=[False, False], kind="mergesort")
    .drop_duplicates("title_key")
    .drop(columns="_reviews")
    .reset_index(drop=True)
)
print(f"steam rows dropped as duplicate titles: {len(steam) - len(steam_one):,}")

hltb release_year == 0 set to missing: 6,723
hltb candidates with a main story time: 42,167


steam rows dropped as duplicate titles: 4,571


In [9]:
def closest_year(pairs):
    """Per Steam game, keep the candidate closest in release year, then the most polled."""
    pairs = pairs.assign(_gap=(pairs["steam_year"] - pairs["hltb_release_year"]).abs())
    return pairs.sort_values(
        ["_gap", "hltb_main_story_polled", "hltb_id"], ascending=[True, False, True], kind="mergesort"
    ).drop_duplicates("steam_appid")

hltb_cols = [c for c in candidates.columns if c != "title_key"]

# Pass 1: exact normalised title. Both sides are unique on title_key, so this is one-to-one.
pass1 = closest_year(steam_one[["steam_appid", "title_key", "steam_year"]].merge(candidates, on="title_key"))

# Pass 2: loose key, only for Steam titles and hltb entries that pass 1 left unused
rest = steam_one.loc[~steam_one["steam_appid"].isin(pass1["steam_appid"]), ["steam_appid", "steam_name", "steam_year", "steam_positive", "steam_negative"]]
rest = rest.assign(loose_key=loose_title(rest["steam_name"]), _reviews=rest["steam_positive"] + rest["steam_negative"])
spare = candidates.loc[~candidates["hltb_id"].isin(pass1["hltb_id"]), hltb_cols]
spare = spare.assign(loose_key=loose_title(spare["hltb_name"]))
pass2 = closest_year(rest[rest["loose_key"].ne("")].merge(spare, on="loose_key"))
# Two Steam editions can reach the same hltb entry. The closer year keeps it, then more reviews.
pass2 = pass2.sort_values(["_gap", "_reviews", "steam_appid"], ascending=[True, False, False], kind="mergesort").drop_duplicates("hltb_id")

matches = pd.concat([pass1.assign(match_pass=1), pass2.assign(match_pass=2)])[["steam_appid", *hltb_cols, "match_pass"]]
assert matches["steam_appid"].is_unique and matches["hltb_id"].is_unique

# No suffixes needed: the source prefixes already make every column unique.
joined = steam_one.merge(matches, on="steam_appid", how="left")
joined["match_pass"] = joined["match_pass"].astype("Int8")
joined["matched_hltb"] = joined["hltb_main_story"].notna()
joined["year_diff"] = joined["steam_year"] - joined["hltb_release_year"]

assert len(joined) == len(steam_one) == config.JOINED_ROWS, "join multiplied rows"
config.verify_appid_fingerprint(joined, config.JOINED_APPID_SHA256, "00_joined")
matched = int(joined["matched_hltb"].sum())
agree = int((joined["year_diff"].abs() <= 1).sum())
print(f"rows in: {len(steam_one):,}")
print(f"matched to hltb: {matched:,} ({matched / len(joined) * 100:.1f}%)")
print(f"  pass 1, exact title: {len(pass1):,}")
print(f"  pass 2, edition fallback: {len(pass2):,}")
print(f"  of which year agrees within 1: {agree:,} ({agree / matched * 100:.1f}% of matches)")
print(f"  of which hltb year is missing: {int((joined['matched_hltb'] & joined['year_diff'].isna()).sum()):,}")
print(f"rows out: {len(joined):,}")

rows in: 121,284
matched to hltb: 21,303 (17.6%)
  pass 1, exact title: 20,826
  pass 2, edition fallback: 477
  of which year agrees within 1: 18,140 (85.2% of matches)
  of which hltb year is missing: 171
rows out: 121,284


In [10]:
# Matched against unmatched, on the variables both groups have
compare = joined.groupby("matched_hltb").agg(
    games=("steam_appid", "size"),
    median_reviews=("steam_positive", "median"),
    median_price=("steam_price", "median"),
    pct_with_metacritic=("steam_metacritic_score", lambda s: (s > 0).mean() * 100),
    median_year=("steam_year", "median"),
)
compare.round(2)

,games,median_reviews,median_price,pct_with_metacritic,median_year
matched_hltb,,,,,
False,99981,2.0,1.99,0.69,2023.0
True,21303,155.0,3.49,16.39,2020.0


In [11]:
joined.to_parquet(config.INTERIM / "00_joined.parquet", index=False)
joined[["steam_appid", "steam_name", "steam_year", "hltb_release_year", "year_diff", "hltb_main_story", "hltb_main_story_polled", "match_pass", "matched_hltb"]].head()

,steam_appid,steam_name,steam_year,hltb_release_year,year_diff,hltb_main_story,hltb_main_story_polled,match_pass,matched_hltb
0,730,Counter-Strike 2,2012,NaN,NaN,NaN,NaN,<NA>,False
1,578080,PUBG: BATTLEGROUNDS,2017,NaN,NaN,NaN,NaN,<NA>,False
2,570,Dota 2,2013,NaN,NaN,NaN,NaN,<NA>,False
3,271590,Grand Theft Auto V Legacy,2015,NaN,NaN,NaN,NaN,<NA>,False
4,105600,Terraria,2011,2011.0,0.0,51.74,689.0,1,True


In [12]:
joined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121284 entries, 0 to 121283
Data columns (total 73 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   steam_appid                       121284 non-null  int64  
 1   steam_name                        121284 non-null  object 
 2   steam_release_date                121284 non-null  object 
 3   steam_estimated_owners            121284 non-null  object 
 4   steam_peak_ccu                    121284 non-null  int64  
 5   steam_required_age                121284 non-null  int64  
 6   steam_price                       121284 non-null  float64
 7   steam_discount                    121284 non-null  int64  
 8   steam_dlc_count                   121284 non-null  int64  
 9   steam_about_the_game              113366 non-null  object 
 10  steam_supported_languages         121284 non-null  object 
 11  steam_full_audio_languages        121284 non-null  o

### Interpretation: The join and its match rate
**Author**: Lee Sin Yang
<!-- Interpret the match rate and the matched-versus-unmatched comparison above. Cover whether the matched subset looks representative of the Steam population, what the direction of any difference implies for the completion ratio results, whether the normalised-title key should be tightened to require the release years to agree, and how this should be reported as a limitation. -->
The match rate is fairly low, due to the long tail of steam games by reviews. Most games on steam have little to no reviews, and thus the ones that survive tend to be more popular. This is an important consideration for future analyses - our data only captures games that perform well enough to appear on HLTB, a form of survivorship bias. 

## 6. Scope: story-driven games matched to HowLongToBeat
*Code: Generated by AI Assistant*

*Written by AI Assistant*: the study population is story-driven games that matched a HowLongToBeat entry with a main-story time. The scope is applied here, before the train and test split, so the holdout is drawn from the same population the report describes and every later stage, EDA included, sees only that population. Every game in scope carries data from both sources.

The scope continues from the end of section 5.

1. Matched. Only games with a HowLongToBeat match are kept. Unmatched games are dropped, not given a review count or completion ratio of 0, because their HowLongToBeat values are unknown rather than zero.
2. Story-driven. "Story-driven" is not a field in either dataset. The rule lives in `classify_story_driven` in `src/is630/features.py` and uses Steam fields only, so it does not depend on the match. A game is story-driven when its Steam `categories` include `Single-player` and either its Steam `tags` include one of `Story Rich`, `Visual Novel`, `Interactive Fiction`, `Narrative`, `Choices Matter` or `Choose Your Own Adventure`, or its Steam `genres` include `Visual Novel`. Tags are matched as whole names. Games with no tags or categories cannot be classified and fall out.

`00_joined.parquet` above keeps every Steam game, so the scope can be re-drawn without re-running the join.

In [13]:
joined["flag_story_driven"] = classify_story_driven(joined)
in_hltb = joined[joined["matched_hltb"]]
scoped = in_hltb[in_hltb["flag_story_driven"]].reset_index(drop=True)

assert len(scoped) == config.SCOPED_ROWS
config.verify_appid_fingerprint(scoped, config.SCOPED_APPID_SHA256, "scope")

has_tags = in_hltb["steam_tags"].notna() & in_hltb["steam_categories"].notna()
print(f"rows in, matched to hltb: {len(in_hltb):,}")
print(f"  missing tags or categories, cannot be classified: {int((~has_tags).sum()):,}")
print(f"  classified, not story-driven: {int((has_tags & ~in_hltb['flag_story_driven']).sum()):,}")
print(f"rows out, story-driven: {len(scoped):,} ({len(scoped) / len(in_hltb) * 100:.1f}% of matched)")

rows in, matched to hltb: 21,303
  missing tags or categories, cannot be classified: 2,665
  classified, not story-driven: 12,037
rows out, story-driven: 6,601 (31.0% of matched)


### Matched against unmatched story-driven games
*Code: Generated by AI Assistant*

*Written by AI Assistant*: the story-driven rule is applied to every Steam game here, so the story-driven games that found no HowLongToBeat match can be compared with the scope, on the Steam variables both have. This is the only place unmatched games appear after the join.

In [14]:
# Matched against unmatched story-driven games, on the Steam variables both have.
# This is the only use of the unmatched games after the join.
story = joined[joined["flag_story_driven"]]
print(f"story-driven games across Steam: {len(story):,}, of which unmatched: {int((~story['matched_hltb']).sum()):,}")
story.assign(review_count=story["steam_positive"] + story["steam_negative"]).groupby("matched_hltb").agg(
    games=("steam_appid", "size"),
    median_reviews=("review_count", "median"),
    pct_zero_reviews=("review_count", lambda s: (s == 0).mean() * 100),
    median_price=("steam_price", "median"),
    pct_with_metacritic=("steam_metacritic_score", lambda s: (s > 0).mean() * 100),
    pct_zero_playtime=("steam_median_playtime_forever", lambda s: (s == 0).mean() * 100),
    median_year=("steam_year", "median"),
).round(2)

story-driven games across Steam: 18,669, of which unmatched: 12,068


,games,median_reviews,pct_zero_reviews,median_price,pct_with_metacritic,pct_zero_playtime,median_year
matched_hltb,,,,,,,
False,12068,17.0,0.43,3.14,1.23,84.60,2022.0
True,6601,342.0,0.27,4.49,20.69,31.18,2020.0


### Interpretation: The scope
**Author**: Lee Sin Yang
<!-- Interpret the scope above. Cover how many matched games the story-driven rule keeps and what share that is, whether the rule is a defensible reading of "story-driven" for the studio's question and which kinds of game it might wrongly include or miss (including games with no tags, which it cannot classify), how many story-driven games are lost by requiring a HowLongToBeat match, and what the matched-versus-unmatched comparison implies about which games the results can speak for. This is the basis for the selection limitation in the report. -->
Scoping down into the games that are matched introduces biased. Median reviews goes from 17 to 342, price goes from 3.14 to 4.49, they have much less empty values, and are generally older. This is a limitation of our analysis as we have no way of knowing the true outcomes of many of the games, so we take the lesser evil of a skewed analysis than none at all.

## 7. Train and test split
*Code: Generated by AI Assistant*

*Written by AI Assistant*: the split is drawn from the scope above. The test set is held out now, before any exploration, and is scored once at the end. Nothing is refitted after that score.

The split is stratified on release year, so both sets span the same years. A plain random split could leave the age mix differing between them, and review counts and playtime both accumulate with age.

A time-based holdout, training on older games and testing on the newest, would be the stricter test of predicting an unreleased game. It is not used here because every accumulating column would then differ systematically between train and test, which would confound the score.

In [15]:
strata = scoped["steam_year"].fillna(-1).astype(int).astype(str)
rare = strata.value_counts()[lambda v: v < 5].index
strata = strata.where(~strata.isin(rare), "rare")

test = scoped.groupby(strata, group_keys=False, sort=True).sample(frac=TEST_FRACTION, random_state=config.SEED)
train = scoped.drop(index=test.index)

assert len(train) == config.TRAIN_ROWS and len(test) == config.TEST_ROWS
assert len(train) + len(test) == len(scoped)
assert not set(train["steam_appid"]) & set(test["steam_appid"])
config.verify_appid_fingerprint(train, config.TRAIN_APPID_SHA256, "00_train")
config.verify_appid_fingerprint(test, config.TEST_APPID_SHA256, "00_test_holdout")

for name, df in [("00_train", train), ("00_test_holdout", test)]:
    df.to_parquet(config.INTERIM / f"{name}.parquet", index=False)
    print(f"{name}: {len(df):,} rows ({len(df) / len(scoped) * 100:.1f}%)")

00_train: 5,281 rows (80.0%)
00_test_holdout: 1,320 rows (20.0%)


*Written by AI Assistant*: `00_test_holdout.parquet` must not be loaded by any notebook except the final scoring step. Every later stage starts from `00_train.parquet`.